# 02 — Narrative Inspection

Load generated narratives from `outputs/generation/<run_id>/narratives.csv`, read them
alongside their ground-truth SHAP values, and manually inspect quality.

In [8]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from dotenv import load_dotenv

load_dotenv('../.env')

from src.config import load_config
from src.storage import list_runs, load_narratives_csv, narratives_csv_path, run_dir

cfg = load_config('../config/default.yaml')

In [9]:
# List all available runs (folders with narratives.csv)
runs = list_runs(f'../{cfg.storage.generation_dir}')
pd.DataFrame(runs)

,run_id,path,n_narratives
0,pilot_run_20260518T135815_bdad28,..\outputs\generation\pilot_run_20260518T13581...,17342
1,pilot_run_20260518T133147_4b0aad,..\outputs\generation\pilot_run_20260518T13314...,577
2,pilot_run_20260518T130411_e9f98c,..\outputs\generation\pilot_run_20260518T13041...,577
3,pilot_run_20260518T123547_d896fe,..\outputs\generation\pilot_run_20260518T12354...,416
4,pilot_run_20260518T122429_35ddd8,..\outputs\generation\pilot_run_20260518T12242...,249
5,pilot_run_20260518T074326_cadfd5,..\outputs\generation\pilot_run_20260518T07432...,83
6,pilot_run_20260518T073857_6d094a,..\outputs\generation\pilot_run_20260518T07385...,48
7,pilot_run_20260518T073743_f2e217,..\outputs\generation\pilot_run_20260518T07374...,48


In [10]:
# Set the run_id you want to inspect
RUN_ID = runs[0]['run_id'] if runs else None
#RUN_ID = "pilot_run_20260518T123547_d896fe"
print('Inspecting run:', RUN_ID)

Inspecting run: pilot_run_20260518T135815_bdad28


In [11]:
import os

csv_path = narratives_csv_path(run_dir(cfg, RUN_ID))
# The outputs/generation/... folder is not under notebooks, but a neighbor folder.
csv_full_path = os.path.abspath(os.path.join('..', csv_path))
narr_df = load_narratives_csv(csv_full_path)
print(f'{len(narr_df)} narratives loaded from {csv_full_path}')
narr_df.head()

300 narratives loaded from c:\xAI\xAI-research\outputs\generation\pilot_run_20260518T135815_bdad28\narratives.csv


,narrative_id,run_id,dataset,instance_id,model_id,prompt_strategy,model_provider,model_name,temperature,max_tokens,pred_proba,pred_label,shap_values_sorted,prompt,narrative_text,error,created_at
0,cef1e67c-a938-4324-abf4-74e25485b24a,pilot_run_20260518T135815_bdad28,adult,1263,llama3-70b,martens,huggingface,meta-llama/Meta-Llama-3-70B-Instruct,0.0,2048,0.269233,0,"[[""marital_status_Non_Married"", 0.097617745], ...",An AI model was used to predict whether a pers...,Here's a plausible story as to why the model p...,NaN,2026-05-18T13:58:39.262004+00:00
1,9ea835a9-0394-43be-aa03-44f158df3184,pilot_run_20260518T135815_bdad28,adult,93,llama3-70b,martens,huggingface,meta-llama/Meta-Llama-3-70B-Instruct,0.0,2048,0.269051,0,"[[""marital_status_Non_Married"", 0.06296883], [...",An AI model was used to predict whether a pers...,Here's a plausible story as to why the model p...,NaN,2026-05-18T13:58:56.769625+00:00
2,13080e4e-232a-4c00-9a84-22bcb744ec77,pilot_run_20260518T135815_bdad28,adult,3317,llama3-70b,martens,huggingface,meta-llama/Meta-Llama-3-70B-Instruct,0.0,2048,0.271540,0,"[[""marital_status_Non_Married"", 0.2029147], [""...",An AI model was used to predict whether a pers...,Here's a plausible story as to why the model p...,NaN,2026-05-18T13:59:22.465849+00:00
3,fc74cdba-0e59-41c3-8c8e-96fd85478050,pilot_run_20260518T135815_bdad28,adult,6021,llama3-70b,martens,huggingface,meta-llama/Meta-Llama-3-70B-Instruct,0.0,2048,0.678034,1,"[[""sex_Male"", 0.11808771], [""marital_status_No...",An AI model was used to predict whether a pers...,Here's a plausible story as to why the model p...,NaN,2026-05-18T13:59:47.833147+00:00
4,cd03211d-31c5-400a-a661-664f4524c256,pilot_run_20260518T135815_bdad28,adult,5694,llama3-70b,martens,huggingface,meta-llama/Meta-Llama-3-70B-Instruct,0.0,2048,0.355665,0,"[[""sex_Male"", 0.14665028], [""occupation_Other""...",An AI model was used to predict whether a pers...,Here's a plausible story as to why the model p...,NaN,2026-05-18T14:00:06.827426+00:00


In [12]:
# Summary: narratives per model × dataset
narr_df.groupby(['model_id', 'dataset']).size().unstack(fill_value=0)

dataset,adult
model_id,
llama3-70b,300


In [13]:
# Inspect a random narrative alongside its SHAP context
import pandas as pd  # avoid re-import warning
from src.data_loader import format_shap_table

sample = narr_df.sample(1).iloc[0]
dataset_cfg = cfg.get_dataset(sample['dataset'])
raw_df = pd.read_csv(f'../{dataset_cfg.path}')
row = raw_df.iloc[sample['instance_id']]

print('=== SHAP VALUES ===')
print(format_shap_table(row, dataset_cfg.shap_col_prefix))
print()
print(f'=== NARRATIVE ({sample["model_id"]} | {sample["dataset"]}) ===')
print(sample['narrative_text'])

=== SHAP VALUES ===
  hours_per_week: +0.0906
  age: +0.0442
  native_country_US: +0.0246
  capital_gain: +0.0080
  marital_status_Non_Married: -0.0039
  relationship_Non_Husband: -0.0044
  workclass_Private: -0.0077
  capital_loss: -0.0090
  race_White: -0.0167
  occupation_Other: -0.1126
  sex_Male: -0.1395

=== NARRATIVE (llama3-70b | adult) ===
Here's a plausible story as to why the model predicted that this person's annual income exceeds $50,000:

Meet John, a 40-year-old married man who works as a machine operator in the private sector. He's a hard worker, putting in 40 hours a week, which is a significant contributor to the model's prediction of his high income. His age, 40, is also a positive factor, suggesting that he's reached a stage in his career where he's likely to be earning a higher salary.

John's native country is the United States, which is another factor that pushes the model's prediction towards a higher income. Additionally, he has no capital losses, which means h

In [14]:
# Browse narratives for a specific model and dataset
MODEL = 'llama3-70b'
DATASET = 'adult'
N = 5  # how many to show

subset = narr_df[
    (narr_df['model_id'] == MODEL) &
    (narr_df['dataset'] == DATASET) &
    (narr_df['error'].fillna('') == '')
].head(N)

for _, row in subset.iterrows():
    print(f'--- Instance {row["instance_id"]} ---')
    print(row['narrative_text'])
    print()

--- Instance 1263 ---
Here's a plausible story as to why the model predicted that this person's annual income is at or below $50,000:

Meet John, a 55-year-old male who works in a private sector job, specifically in an "other" occupation, which might include machine operation, transportation, or farming. He is non-married, having never been married, divorced, or separated, and is not a husband in his current relationship. John is a hard worker, putting in 51 hours per week, and is a native of the United States.

Despite his diligence, the model predicts that John's income is at or below $50,000. Let's explore the factors that contributed to this prediction.

On the positive side, John's non-married status and non-husband relationship status both pushed his income prediction upward. Perhaps being single or not being a husband allows John to focus more on his career, leading to higher earning potential. Additionally, being male and working in a private sector job also contributed positiv